# Model B — Language Understanding

Extracts structured clinical information from a patient transcript using Gemini.
No training data required — uses Gemini with a fixed schema and prompt guardrails.

**Output schema:**
```json
{
  "chief_complaint": "",
  "duration": "",
  "severity": "",
  "body_part": "",
  "associated_symptoms": [],
  "red_flags_present": null,
  "additional_observations": ""
}
```

In [ ]:
!pip install -q -U google-generativeai

In [ ]:
import json
import re
from dataclasses import dataclass, field, asdict
from typing import List, Optional

import google.generativeai as genai
from google.colab import userdata

genai.configure(api_key=userdata.get("GEMINI_API_KEY"))

model = genai.GenerativeModel(
    model_name="gemini-1.5-flash",
    generation_config=genai.types.GenerationConfig(temperature=0.0, max_output_tokens=512),
    safety_settings=[
        {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_ONLY_HIGH"},
    ],
)

In [ ]:
@dataclass
class ClinicalExtraction:
    chief_complaint:         str            = ""
    duration:                str            = ""
    severity:                str            = ""
    body_part:               str            = ""
    associated_symptoms:     List[str]      = field(default_factory=list)
    red_flags_present:       Optional[bool] = None
    additional_observations: str            = ""


EMPTY_SCHEMA = {
    "chief_complaint":         "",
    "duration":                "",
    "severity":                "",
    "body_part":               "",
    "associated_symptoms":     [],
    "red_flags_present":       None,
    "additional_observations": "",
}

RED_FLAG_TERMS = [
    "can't breathe", "cannot breathe", "chest pain", "chest tightness",
    "unconscious", "fainted", "collapsed", "severe bleeding", "coughing blood",
    "seizure", "convulsion", "sudden vision loss", "paralysis", "can't move",
    "guhumeka", "amaraso", "imitsi", "kunanirwa",
]

In [ ]:
SYSTEM_PROMPT = """\
You are a clinical information extraction assistant.
Extract observable facts from the patient transcript and populate the JSON schema.

Rules:
- Output valid JSON only. No markdown, no extra text.
- Leave fields empty if information is missing. Never guess.
- No diagnosis, no medical advice, no new fields.
- red_flags_present: true if breathing difficulty, chest pain, loss of consciousness,
  heavy bleeding, seizure, or sudden paralysis is mentioned. Otherwise false or null.
- The transcript may be in English, Kinyarwanda, or both.

Schema:
{schema}
"""


def build_prompt(transcript: str) -> str:
    return f"Transcript:\n{transcript.strip()}\n\nReturn the populated JSON schema only."

In [ ]:
def _parse(raw: str) -> dict:
    cleaned = re.sub(r"```(?:json)?\s*", "", raw).strip().rstrip("`")
    match   = re.search(r"\{.*\}", cleaned, re.DOTALL)
    if not match:
        raise ValueError("No JSON in response.")
    return json.loads(match.group(0))


def _validate(raw_dict: dict) -> ClinicalExtraction:
    d = {k: raw_dict.get(k, v) for k, v in EMPTY_SCHEMA.items()}

    for key in ["chief_complaint", "duration", "severity", "body_part", "additional_observations"]:
        if not isinstance(d[key], str):
            d[key] = str(d[key]) if d[key] else ""

    if not isinstance(d["associated_symptoms"], list):
        d["associated_symptoms"] = [d["associated_symptoms"]] if d["associated_symptoms"] else []

    if d["red_flags_present"] not in (True, False, None):
        d["red_flags_present"] = None

    # Keyword cross-check to catch any missed red flags
    all_text = " ".join([
        d["chief_complaint"], d["additional_observations"],
        " ".join(d["associated_symptoms"]),
    ]).lower()
    if any(term in all_text for term in RED_FLAG_TERMS):
        d["red_flags_present"] = True

    return ClinicalExtraction(**d)


def extract(transcript: str) -> dict:
    """
    Extract structured clinical information from a transcript.
    Returns a dict with 'extraction_dict' and 'extraction' (dataclass).
    """
    if len(transcript.strip()) < 10:
        empty = ClinicalExtraction()
        return {"extraction": empty, "extraction_dict": asdict(empty), "success": False, "error": "Transcript too short."}

    system = SYSTEM_PROMPT.format(schema=json.dumps(EMPTY_SCHEMA, indent=2))
    chat   = model.start_chat(history=[
        {"role": "user",  "parts": [system]},
        {"role": "model", "parts": ["Understood. JSON only, no diagnosis."]},
    ])

    try:
        response   = chat.send_message(build_prompt(transcript))
        extraction = _validate(_parse(response.text))
        return {"extraction": extraction, "extraction_dict": asdict(extraction), "success": True, "error": None}
    except Exception as e:
        empty = ClinicalExtraction()
        return {"extraction": empty, "extraction_dict": asdict(empty), "success": False, "error": str(e)}

---
## Usage

### Option A — From a transcript string

In [ ]:
transcript = "I've had a headache for two days. It's on the right side and quite severe."

result = extract(transcript)
print(json.dumps(result["extraction_dict"], indent=2))

### Option B — From Model A output

In [ ]:
# model_a_result = transcribe_file("audio.wav")   # from Model A
# result = extract(model_a_result["full_text"])
# print(json.dumps(result["extraction_dict"], indent=2))